# Setup: timestral-vol on Colab

This notebook prepares a Colab runtime to run the `timestral-vol` project: it mounts Google Drive for persistent caching, clones the repo, installs dependencies, and runs a smoke test that downloads data, builds the volatility target, and gets a zero-shot Timestral forecast.

**Before running:** `Runtime > Change runtime type > T4 GPU`.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATA = "/content/drive/MyDrive/timestral-vol-data"
for sub in ["cache", "hf_cache", "checkpoints", "results"]:
    os.makedirs(f"{DATA}/{sub}", exist_ok=True)

DATA


In [ ]:
import os

REPO_DIR = "/content/timestral-vol"
REPO_URL = "https://github.com/MachaGotJuice/timestral-vol.git"

if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

!pip install -q -r requirements.txt


In [ ]:
import os
import sys

os.environ["HF_HOME"] = f"{DATA}/hf_cache"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

sys.path.insert(0, REPO_DIR)


## Smoke test

Download SPY, build the Garman-Klass variance and the 30-day log target, load the Timestral-5M model, and forecast 30 steps ahead from the last 1024 values of the log target.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.data import download_ohlc, garman_klass, target_30d
from src.timestral_utils import forecast_quantiles, load_model

df = download_ohlc("SPY", cache_dir=f"{DATA}/cache")
daily_var = garman_klass(df)
log_target = target_30d(daily_var, window=30)

context = log_target.values[-1024:]
contexts = context[None, :]  # (B=1, T)

model = load_model(size="5M")
quantiles, quantile_levels = forecast_quantiles(model, contexts, horizon=30)

print("context shape:", context.shape)
print("quantiles shape:", quantiles.shape)
print("quantile_levels:", quantile_levels)

median_idx = int(np.argmin(np.abs(quantile_levels - 0.5)))
lower_idx = int(np.argmin(np.abs(quantile_levels - 0.10)))
upper_idx = int(np.argmin(np.abs(quantile_levels - 0.90)))

median = quantiles[0, :, median_idx]
lower = quantiles[0, :, lower_idx]
upper = quantiles[0, :, upper_idx]
print("forecast median:", median)

history_x = np.arange(len(context))
forecast_x = np.arange(len(context), len(context) + len(median))

plt.figure(figsize=(12, 5))
plt.plot(history_x, context, label="History (log target_30d)")
plt.plot(forecast_x, median, label="Forecast median")
plt.fill_between(forecast_x, lower, upper, alpha=0.3, label="10-90% band")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
